# Module 7 — Evaluating Answers

Part of the RAG Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

**Where this module starts**
- RAG (retrieval-augmented generation) answers a question in two steps:
  1. A search program finds the few texts in a collection that match the question.
  2. A language model, the kind of program behind chat assistants, reads those texts and writes the answer.
- A RAG system is built from five parts:
  1. Chunk: split each document into short pieces, called chunks
  2. Embed: turn each chunk into an embedding, a list of numbers that an embedding model makes from a text, so that texts with similar meaning get similar embeddings
  3. Store: keep every chunk's embedding in one table
  4. Retrieve: for each question, return the chunks that match it best
  5. **Generate**: the language model reads those chunks and writes the answer
- Everything the language model gets, the question included, is called the prompt. In part 5, the prompt holds the returned chunks, each with a number, then the question, then instructions for the answer.

**This module's point:** it builds no new part. It grades the answers that part 5 writes, with two measures:
- Correctness: does the answer give the same answer as a correct answer known in advance? Only a test question, written with its correct answer, can be graded this way.
- Faithfulness: is every claim in the answer stated in the returned chunks?

The two measures can disagree. An answer can be correct but not faithful: it gives the correct answer, and adds a claim that no chunk holds. An answer can also be faithful but not correct: every claim is in the chunks, but the chunks point to a wrong answer.

An answer can be ruled on in three ways:
- By hand: a person reads the answer. This is the most reliable way, but it is slow.
- By a word-match program: a program checks whether the correct answer appears in the answer as a whole word, ignoring capital letters. This is fast, but the program reads words, not meaning.
- By a language model as judge: a language model gets the answer and a question about it, and replies yes or no. This is fast and reads meaning, but the judge can be wrong.

The question is how often the word match and the judge agree with the hand check on correctness, and how often the judge does on faithfulness. The word match is not used for faithfulness: no single word shows whether a claim is in the chunks. A last question follows: when an answer is not right, did the returned chunks hold the facts the question asks about?

<u>Note</u>: the whole RAG system stays the same throughout this module, so only the grading changes:
- Part 1, chunk: each news article is split into its paragraphs, one chunk per paragraph.
- Part 2, embed: an existing embedding model, nomic-embed-text.
- Part 3, store: has no setting to choose.
- Part 4, retrieve: hybrid search, which combines a meaning search (it compares the question's embedding with each chunk's embedding) and a keyword search (it scores the words a chunk shares with the question). Each chunk gets a hybrid score = 1 / (60 + its rank in the meaning search) + 1 / (60 + its rank in the keyword search), where rank 1 is the best match and 60 is a standard setting. The search returns the chunks with the highest scores while their words stay at or below 400 in total, a number chosen for this playbook, and always at least 1 chunk.
- Part 5, generate: the language model llama3.2:3b, a small, free model that runs on this computer. Its instructions ask for one or two sentences, and set three rules:
  1. Use only the chunks.
  2. After each sentence of the answer, write the numbers of the chunks it comes from in square brackets, such as [2]. This is called citing.
  3. If the chunks do not hold the answer, reply only *"Not enough information."*

**What this module measures**

The test questions come from a public dataset called MultiHop-RAG, of questions about news articles. Each question was written from two to four articles. For each question, the dataset gives the correct answer and lists the sentences in those articles that support it. This module uses two kinds of question:
1. Name questions: the correct answer is a name, such as a company or a person.
2. Yes/no questions: the correct answer is yes or no.

The module runs four steps:
1. The answers and the hand check: the language model answers the name questions and the yes/no questions. The hand check rules every answer *right* or *not right*, and every answer to a name question *faithful* or *not faithful*. **[in real life]** A person makes the hand check. **[only in this module]** A larger language model made the hand check, by reading each answer next to its chunks, and no person has checked its rulings. Faithfulness is ruled on the name answers only: a yes/no answer mixes facts with its conclusion, the yes or no, which no chunk states, so ruling it would need a rule for which parts of the answer count.
2. Correctness: the word match and the judge rule every answer *right* or *not right*. Each is compared with the hand check, for each kind of question.
3. Faithfulness: the judge rules every answer to a name question *faithful* or *not faithful*, and is compared with the hand check.
4. Whose fault: for each answer, did the search return a listed sentence from every article the question was written from? The answers that are not right are compared with the answers that are right.

**Two tags in this module**
- **[in real life]** marks a step that happens in a real system, in a test before the system goes live: the analyst grades the answers to questions that come with a correct answer.
- **[only in this module]** marks a step that a real system would not do, chosen to make this module simpler or faster.

**What this module produces**
1. The two prompts the judge gets: one for correctness, one for faithfulness.
2. For each kind of question, how often the word match and the judge agree with the hand check on correctness, next to a blind checker that gives every answer the same ruling.
3. How often the judge agrees with the hand check on faithfulness, next to the same kind of checker.
4. The answers, right and not right, split by whether the search returned a listed sentence from every article.

## From the curriculum

> ## <u>Module 7 — Evaluating Answers</u>
>
> **Concepts**
> - Correctness: does the answer match the labelled answer?
> - Faithfulness: does the answer use only what was retrieved?
> - Using a language model as a judge, and checking the judge by hand
> - Separating a retrieval failure from a generation failure
>
> **Worked example:** a sample of answers ruled on both ways, with the judge's
> errors counted.

In this module, the labelled answer is called the correct answer, the two ways are the word match and the judge, each checked against the hand check, a retrieval failure is called a search failure, and a generation failure a language model failure. The word match and the judge both rule correctness. Only the judge also rules faithfulness, because no single word shows whether a claim is in the chunks.

## Lesson

### Correctness and faithfulness

Correctness compares the answer with the dataset's correct answer. It needs a correct answer, so it can only be measured on test questions that come with one, never on a live user's question.

Faithfulness compares the answer with the returned chunks. It needs no correct answer: only the answer and the chunks the language model read. A claim that is in no chunk came from somewhere else: from the question's own wording, or from the language model's training text, the text it learned from long before the question. Such a claim cannot be checked against the collection, even when it happens to be true.

### The language model as judge

The judge is a language model given a prompt that asks one question about an answer, with a one-word reply: yes or no. This module uses two judge prompts, printed by the code below:
- Correctness: the question, the dataset's correct answer, and the answer to rule on. "Does the answer to rule on give the same answer as the correct answer?"
- Faithfulness: the returned chunks and the answer to rule on. "Is every claim in the answer to rule on stated in the chunks above?"

**[only in this module]** The judge is llama3.2:3b, the same language model that wrote the answers, because it is the only language model this playbook runs. A real system would use a larger model as judge, and still check the judge by hand, as this module does.

### Checking a checker

The hand check gives each answer its ruling. The word match and the judge are each checked against it:

Agreement with the hand check = answers where the checker's ruling matches the hand check's ruling / answers

A checker can be wrong in two directions, and the code counts each:
- it rules *right* an answer the hand check ruled *not right*
- it rules *not right* an answer the hand check ruled *right*

An agreement count alone does not show that a checker reads well. When most answers get the same hand ruling, a checker that always gives that ruling agrees on most answers without reading any. So the word match and the judge are each also compared with a blind checker: it rules every answer with the hand check's more common ruling. The blind checker needs the hand check's rulings, so it is never used to rule answers: in real life, it is used only as a bar for testing the word match and the judge. The word match or the judge is useful only if it agrees with the hand check more often than the blind checker.

The code below loads the tools and prints the two judge prompts, with their slots in braces. `common` is this playbook's shared helper file: it loads the data, calls the models, and prints tables.

In [1]:
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import common
from common import show

BUDGET = 400                                       # words returned per question (part 4)
K1, B, RRF = 1.2, 0.75, 60                         # standard settings, not chosen here: K1 and B of the keyword search, and RRF, the 60 in the hybrid score
SAMPLE = 30                                        # questions of each kind, chosen for run time
MAX_TOKENS = 120                                   # longest answer, in tokens (word-pieces, about 3/4 of a word each), chosen for run time
tidy = lambda text: " ".join(text.split())       # one space between words, no line breaks

ANSWER_PROMPT = ("Here are chunks of news articles:\n\n{chunks}\n\nQuestion: {question}\n"
                 "Answer in one or two sentences, using only the chunks above. "
                 "After each sentence of your answer, write the numbers of the chunks it comes from in square brackets, such as [2]. "
                 "If the chunks do not hold the answer, reply only: Not enough information.")
JUDGE_CORRECT = ("Question: {question}\nCorrect answer: {correct}\nAnswer to rule on: {answer}\n\n"
                 "Does the answer to rule on give the same answer as the correct answer? Reply with one word: yes or no.")
JUDGE_FAITHFUL = ("Here are chunks of news articles:\n\n{chunks}\n\nAnswer to rule on: {answer}\n\n"
                  "Is every claim in the answer to rule on stated in the chunks above? "
                  "The numbers in square brackets are chunk numbers: ignore them. Reply with one word: yes or no.")
for name, prompt in [("correctness", JUDGE_CORRECT), ("faithfulness", JUDGE_FAITHFUL)]:
    print(f"Judge prompt, {name}:")
    print("  " + prompt.replace("\n", "\n  "))
    print()

Judge prompt, correctness:
  Question: {question}
  Correct answer: {correct}
  Answer to rule on: {answer}
  
  Does the answer to rule on give the same answer as the correct answer? Reply with one word: yes or no.

Judge prompt, faithfulness:
  Here are chunks of news articles:
  
  {chunks}
  
  Answer to rule on: {answer}
  
  Is every claim in the answer to rule on stated in the chunks above? The numbers in square brackets are chunk numbers: ignore them. Reply with one word: yes or no.



## The data: the collection and the test questions

The dataset is MultiHop-RAG (Tang and Yang, 2024):
- 609 news articles, published between September and December 2023, each labelled with one category, such as technology or sports
- 2,556 questions about those articles. Each question needs facts from two to four articles, and the dataset lists the sentences of those articles that support the answer.

**[only in this module]** The collection is the 172 technology articles, not all 609: making the embeddings of all 609 articles' chunks would take hours on this computer.

**[only in this module]** Writing an answer takes the language model about 13 seconds on this computer, and each answer also gets one or two judge rulings. So the module uses 30 questions of each kind, a number chosen to keep a first run under about half an hour, drawn at random with a fixed seed, a starting number that makes the same draw come back on every run. Only questions whose listed sentences all come from technology articles are drawn:
1. Name questions: 30 of the dataset's inference questions. An inference question asks which company or person several articles describe, so its correct answer is a name.
2. Yes/no questions: 30 of the dataset's comparison and temporal questions whose correct answer is yes or no. A comparison question asks whether two articles say the same thing, and a temporal question asks about the order of events or reports.

30 questions per kind is small: on another 30 drawn the same way, every count in this module could differ.

In [2]:
corpus = common.load_corpus()
questions = common.load_questions()
articles = corpus[corpus["category"] == "technology"].reset_index(drop=True)
titles = set(articles["title"])
chunks = [tidy(p) for body in articles["body"] for p in body.split("\n\n") if p.strip()]
words_of_chunk = np.array([len(c.split()) for c in chunks])

in_technology = questions["evidence_list"].map(lambda evidence: bool(evidence) and {e["title"] for e in evidence} <= titles)
pool_name = questions[in_technology & (questions["question_type"] == "inference_query")]
pool_yes_no = questions[in_technology & questions["question_type"].isin(["comparison_query", "temporal_query"])
                        & questions["answer"].str.lower().isin(["yes", "no"])]
name_q = pool_name.sample(SAMPLE, random_state=common.SEED)
yes_no_q = pool_yes_no.sample(SAMPLE, random_state=common.SEED)

print(f"Technology articles, the collection = {len(articles):,}")
print(f"Chunks, one per paragraph = {len(chunks):,}")
print()
print(f"Name questions whose listed sentences all come from technology articles = {len(pool_name):,}, of which drawn = {SAMPLE}")
print(f"Yes/no questions whose listed sentences all come from technology articles = {len(pool_yes_no):,}, of which drawn = {SAMPLE}")

Technology articles, the collection = 172
Chunks, one per paragraph = 7,137

Name questions whose listed sentences all come from technology articles = 371, of which drawn = 30
Yes/no questions whose listed sentences all come from technology articles = 385, of which drawn = 30


**[in real life]** The search runs once per question, and the language model answers from the returned chunks. The code below sets up the search exactly as described in the note at the top, then writes all 60 answers. `common` saves every answer in a file on this computer, so a re-run reads the answers back instead of writing them again.

In [3]:
words_in = lambda text: re.findall(r"[a-z0-9]+", text.lower())


def keyword_search(chunks):
    # BM25 (Best Matching 25), the keyword search's score. K1 and B limit how much repeated words and long chunks raise it. Returns a function: question -> one score per chunk.
    tokens = [words_in(c) for c in chunks]
    n = len(chunks)
    length = np.array([len(t) for t in tokens])
    holding = defaultdict(lambda: ([], []))      # word -> (chunks that hold it, times in each)
    for i, words in enumerate(tokens):
        for w, times in Counter(words).items():
            holding[w][0].append(i)
            holding[w][1].append(times)
    holding = {w: (np.array(rows), np.array(times, dtype=float)) for w, (rows, times) in holding.items()}
    length_part = K1 * (1 - B + B * length / length.mean())

    def scores(question):
        total = np.zeros(n)
        for w in dict.fromkeys(words_in(question)):
            if w in holding:
                rows, times = holding[w]
                rarity_weight = np.log(1 + (n - len(rows) + 0.5) / (len(rows) + 0.5))
                total[rows] += rarity_weight * (times * (K1 + 1)) / (times + length_part[rows])
        return total
    return scores


def ranks_of(scores):
    # Rank 1 = highest score. Ties keep the collection's order.
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


unit = lambda m: m / np.linalg.norm(m, axis=1, keepdims=True)
store = unit(common.embed(chunks, prefix="search_document: "))   # nomic-embed-text expects this prefix before a passage, and "search_query: " before a question
keyword = keyword_search(chunks)


def retrieve(question):
    # The chunks returned for a question, best first: up to 400 words in total, and at least 1 chunk.
    meaning_ranks = ranks_of(store @ unit(common.embed([question], prefix="search_query: "))[0])
    hybrid = 1 / (RRF + meaning_ranks) + 1 / (RRF + ranks_of(keyword(question)))
    order = np.argsort(-hybrid, kind="stable")
    k = max(1, int((np.cumsum(words_of_chunk[order]) <= BUDGET).sum()))
    return [chunks[i] for i in order[:k]]


numbered = lambda returned: "\n\n".join(f"[{i}] {c}" for i, c in enumerate(returned, start=1))
for table in [name_q, yes_no_q]:
    table["returned"] = table["query"].map(retrieve)
    table["answer_text"] = [common.generate(ANSWER_PROMPT.format(chunks=numbered(r), question=q), max_tokens=MAX_TOKENS)
                            for q, r in zip(table["query"], table["returned"])]

## Step 1 of 4: the answers and the hand check

**What does each answer say, and which answers are right?**

**[in real life]** A person reads every answer and rules it. Correctness: *right* when the answer gives the dataset's correct answer, *not right* otherwise, which includes an answer that declines, gives neither yes nor no, or gives both. Faithfulness, for the name questions only: *faithful* when every claim in the answer is stated in the returned chunks.

The hand check below was written by reading all 60 answers and their chunks while this module was built. To keep the code short, it lists only the answers ruled *not right* or *not faithful*, each with its reason. Every answer it does not list is *right*, and every name answer it does not list is *faithful*. The output prints two examples first, one of each kind, picked by hand to show how each kind of answer is written.

In [4]:
# Question number (1-30 within each kind) -> reason. Every answer not listed is ruled right (and, for name questions, faithful).
NOT_RIGHT = {
    ("name", 5): "the answer names Google, where the correct answer is Apple",
    ("name", 6): "the answer names Meta, where the correct answer is Google",
    ("name", 13): "the answer names Google, where the correct answer is Apple",
    ("yes/no", 1): "the answer says the two articles suggest a similar impact: a yes, where the correct answer is no",
    ("yes/no", 8): "the answer declines: it says the chunks do not let it decide",
    ("yes/no", 11): "the answer says the issues remained consistent, so no change: a yes, where the correct answer is no",
    ("yes/no", 13): "the answer says The Verge article does not focus on the withdrawals: a no, where the correct answer is yes",
    ("yes/no", 14): "the answer gives neither yes nor no",
    ("yes/no", 15): "the answer says the Engadget article does not suggest a launch: a no, where the correct answer is yes",
    ("yes/no", 16): "the answer says the portrayal shifted, an inconsistency: a yes, where the correct answer is no",
    ("yes/no", 17): "the answer gives neither yes nor no",
    ("yes/no", 22): "the answer gives both: consistent, then no consistency found",
    ("yes/no", 24): "the answer says both parts hold: a yes, where the correct answer is no",
    ("yes/no", 26): "the answer gives both: unchanged, then a possible shift",
    ("yes/no", 27): "the answer says both parts hold: a yes, where the correct answer is no",
    ("yes/no", 28): "the answer gives neither yes nor no",
    ("yes/no", 30): "the answer says Engadget did not report the discount, so it failed to: a yes, where the correct answer is no",
}
# Name questions: question number -> the claim in the answer that no returned chunk holds.
NOT_FAITHFUL = {
    3: "that Google influences local search rankings with specific criteria",
    4: "that Apple enforces uniform terms through its store and payment system",
    5: "that there is no valid alternative to Google as a search engine",
    6: "that Meta faces scrutiny for its market practices",
    8: "that The Verge and TechCrunch covered Google",
    12: "that The Verge covered a trial with Google as the antagonist",
    13: "that Google enforces uniform terms through its store and payment system",
    14: "that Snapchat was used for the indirect delivery of pills",
    15: "that Google is criticized for its AI model's architecture",
    16: "that Google manipulates search results to maximize ad revenue",
    18: "that Eddy Cue called Apple the only viable option for search",
    19: "that Apple enforces uniform terms on developers, OEMs and carriers",
    20: "that The Age and The Verge reported the claims",
    23: "that Google provided extensive documentation to counter the allegations",
    24: "that Google is at the center of allegations of potential foul play",
    25: "that a major tech competitor deemed Google the only valid search engine",
    28: "that Google provided extensive evidence to counter claims",
    30: "that Google held an annual hardware event and compared Gemini Pro to GPT-3.5",
}
for kind, table in [("name", name_q), ("yes/no", yes_no_q)]:
    numbers = range(1, len(table) + 1)
    table["hand: correct"] = ["not right" if (kind, n) in NOT_RIGHT else "right" for n in numbers]
name_q["hand: faithful"] = ["not faithful" if n in NOT_FAITHFUL else "faithful" for n in range(1, SAMPLE + 1)]


def show_answer(kind, table, n):
    row = table.iloc[n - 1]
    print(f"{kind} question {n}: {row['query']}")
    print(f"  Correct answer: {row['answer'].lower() if kind == 'yes/no' else row['answer']}")
    print(f"  Answer: {tidy(row['answer_text'])}")
    print(f"  Hand check: {row['hand: correct']}" + (f", because {NOT_RIGHT[(kind, n)]}" if (kind, n) in NOT_RIGHT else ""))


show_answer("name", name_q, 21)
print()
show_answer("yes/no", yes_no_q, 23)
print()
print("=" * 60)
print()
rows = [{"kind of question": kind, "questions": str(len(t)), "hand check: right": str((t["hand: correct"] == "right").sum()),
         "hand check: not right": str((t["hand: correct"] == "not right").sum())} for kind, t in [("name", name_q), ("yes/no", yes_no_q)]]
rows.append({"kind of question": "total", "questions": str(2 * SAMPLE),
             "hand check: right": str(sum(int(r["hand check: right"]) for r in rows)),
             "hand check: not right": str(sum(int(r["hand check: not right"]) for r in rows))})
print("Hand check, correctness:")
print()
show(pd.DataFrame(rows))
print()
print(f"Hand check, faithfulness of the {SAMPLE} name answers: faithful = {(name_q['hand: faithful'] == 'faithful').sum()}, not faithful = {len(NOT_FAITHFUL)}")
right_not_faithful = ((name_q["hand: correct"] == "right") & (name_q["hand: faithful"] == "not faithful")).sum()
print(f"  of the {len(NOT_FAITHFUL)} not faithful, right = {right_not_faithful}")

name question 21: What is the name of the company that was discussed on TechCrunch for removing AI-created songs and introducing an AI-powered DJ feature, and was also mentioned on The Verge for achieving its first operating profit in a year, leading to a significant rise in its stock value?
  Correct answer: Spotify
  Answer: The company is Spotify, which removed AI-created songs and introduced an AI-powered DJ feature, and achieved its first operating profit in a year, leading to a significant rise in its stock value. [1] [2]
  Hand check: right

yes/no question 23: Does the TechCrunch article on generative AI in the enterprise suggest that CIOs are more cautious in their AI adoption strategy compared to the belief of business leaders mentioned in another TechCrunch article, who think AI will be essential for all businesses within five years?
  Correct answer: yes
  Answer: The article suggests that CIOs are indeed more cautious in their AI adoption strategy, as only 4% of respondent

**Observation:** the name answer names *Spotify* in its first words, so a reader finds the answer at once. The yes/no answer never writes *yes* or *no*: it restates the question as a statement, *"the article suggests that CIOs are indeed more cautious"*, and the reader has to work out that this means yes.

**Observation:** the language model gets far more name questions right than yes/no questions. 15 name answers are *right* but *not faithful*: a right answer is not proof of a faithful one.

## Step 2 of 4: correctness by word match and by judge

**Does each checker agree with the hand check on which answers are right?**

**[in real life]** The word match and the judge each rule every answer *right* or *not right*. Each ruling is then compared with the hand check's ruling.
- The word match looks for the correct answer as a whole word. On a yes/no question, *not* and *know* therefore do not count as *no*.
- The judge replies yes or no. Its ruling is *right* when its reply starts with *yes*.

The output counts the judge's replies, to check that every reply is a plain yes or no.

In [5]:
def word_match(answer, correct):
    return "right" if re.search(r"\b" + re.escape(correct.lower()) + r"\b", answer.lower()) else "not right"


says_yes = lambda reply: reply.strip().lower().startswith("yes")
judge_replies = []
for table in [name_q, yes_no_q]:
    table["word match: correct"] = [word_match(a, c) for a, c in zip(table["answer_text"], table["answer"])]
    replies = [common.generate(JUDGE_CORRECT.format(question=q, correct=c, answer=a), max_tokens=10)
               for q, c, a in zip(table["query"], table["answer"], table["answer_text"])]
    judge_replies += replies
    table["judge: correct"] = ["right" if says_yes(r) else "not right" for r in replies]


def against_hand(table, checker, measure, values):
    # Answers counted by ruling: rows = the hand check, columns = the checker.
    rows = []
    for v in values + ["total"]:
        part = table if v == "total" else table[table[f"hand: {measure}"] == v]
        cells = {f"{checker}: {c}": str((part[f"{checker}: {measure}"] == c).sum()) for c in values}
        rows.append({"": v if v == "total" else f"hand check: {v}", **cells, "total": str(len(part))})
    return pd.DataFrame(rows)


def agreement(table, checker, measure):
    # Agreement = answers where both rulings match, shown as the sum of the matching cells of the table.
    match = table[f"{checker}: {measure}"] == table[f"hand: {measure}"]
    parts = [f"{(match & (table[f'hand: {measure}'] == v)).sum()} {v}{'s' if v.endswith('right') else ''}" for v in sorted(table[f"hand: {measure}"].unique(), key=lambda v: v.startswith("not"))]
    return match.sum(), (" + ".join(parts) + " = " if len(parts) > 1 else "") + f"{match.sum()} / {len(table)}"


def compare(kind, table):
    common_ruling = table["hand: correct"].value_counts().idxmax()
    print(f"{kind} questions, blind checker, rules every answer {common_ruling}: agreement with the hand check = {(table['hand: correct'] == common_ruling).sum()} / {len(table)}")
    print()
    for checker in ["word match", "judge"]:
        agree, share = agreement(table, checker, "correct")
        print(f"{kind} questions, {checker}: agreement with the hand check = {share}")
        print()
        show(against_hand(table, checker, "correct", ["right", "not right"]))
        print()


print("Judge replies, correctness: " + ", ".join(f"{r} = {c}" for r, c in Counter(r.strip().rstrip(".") for r in judge_replies).items()))

Judge replies, correctness: Yes = 42, No = 18


The output shows how often each checker agrees with the hand check on the name questions: first the blind checker, then the word match, then the judge.

In [6]:
compare("name", name_q)

name questions, blind checker, rules every answer right: agreement with the hand check = 27 / 30

name questions, word match: agreement with the hand check = 27 rights + 3 not rights = 30 / 30

                       word match: right  word match: not right  total
hand check: right                     27                      0     27
hand check: not right                  0                      3      3
total                                 27                      3     30

name questions, judge: agreement with the hand check = 22 rights + 3 not rights = 25 / 30

                       judge: right  judge: not right  total
hand check: right                22                 5     27
hand check: not right             0                 3      3
total                            22                 8     30



**Observation:** on the name questions, most answers are *right*, so ruling every answer *right* already agrees on 27 of 30. The word match agrees on every answer: a name answer writes the name, so finding the word is enough. The judge agrees less often than ruling every answer *right*: it ruled 5 right answers *not right*. Its reply is one word, so the reason is unknown.

The output shows the same three checkers on the yes/no questions.

In [7]:
compare("yes/no", yes_no_q)

yes/no questions, blind checker, rules every answer right: agreement with the hand check = 16 / 30

yes/no questions, word match: agreement with the hand check = 3 rights + 12 not rights = 15 / 30

                       word match: right  word match: not right  total
hand check: right                      3                     13     16
hand check: not right                  2                     12     14
total                                  5                     25     30

yes/no questions, judge: agreement with the hand check = 14 rights + 8 not rights = 22 / 30

                       judge: right  judge: not right  total
hand check: right                14                 2     16
hand check: not right             6                 8     14
total                            20                10     30



**Observation:** on the yes/no questions, the word match agrees with the hand check less often than ruling every answer *right*. It ruled 13 right answers *not right*, because those answers never write *yes* or *no*. It also ruled 2 answers *right* that are not: each holds the word *no* in another sense, such as *"There is no mention"*. The judge agrees more often than the blind checker: reading meaning helps here, but it still ruled 6 answers *right* that are not.

The output below prints every answer on which the word match and the hand check disagree, to show the two kinds of error: 15 answers (0 name answers + 15 yes/no answers).

In [8]:
for n, (_, row) in enumerate(yes_no_q.iterrows(), start=1):
    if row["word match: correct"] != row["hand: correct"]:
        print(f"yes/no question {n}: {row['query']}")
        print(f"  Correct answer: {row['answer'].lower()}. Word match: {row['word match: correct']}. Hand check: {row['hand: correct']}")
        print(f"  Answer: {tidy(row['answer_text'])}")
        print()

yes/no question 2: Does the TechCrunch article reporting on the EU's investigation into X over alleged disinformation disagree with the TechCrunch article warning Elon Musk's X (formerly Twitter) about illegal content and disinformation in terms of the type of content being circulated, with both mentioning the circulation of false information and manipulated content?
  Correct answer: no. Word match: not right. Hand check: right
  Answer: The TechCrunch article reporting on the EU's investigation into X over alleged disinformation does not disagree with the EU's warning about the circulation of false information and manipulated content, as both mention the circulation of "illegal content and disinformation" [1, 3]. [2]

yes/no question 3: Between the report from The Verge on Apple's defense of its Google Search deal published on September 26, 2023, and the class action antitrust suit against Google reported by TechCrunch on December 15, 2023, was there a change in the portrayal of Goog

The output below prints every answer on which the judge and the hand check disagree, to look for what the judge's errors have in common: 13 answers (5 name answers + 8 yes/no answers).

In [9]:
for kind, table in [("name", name_q), ("yes/no", yes_no_q)]:
    disagree = (table["judge: correct"] != table["hand: correct"]).sum()
    print(f"{kind.capitalize()} answers on which the judge and the hand check disagree = {disagree}")
    print()
    for n, (_, row) in enumerate(table.iterrows(), start=1):
        if row["judge: correct"] != row["hand: correct"]:
            print(f"{kind} question {n}: {row['query']}")
            print(f"  Correct answer: {row['answer'].lower() if kind == 'yes/no' else row['answer']}")
            print(f"  Answer: {tidy(row['answer_text'])}")
            print(f"  Judge: {row['judge: correct']}. Hand check: {row['hand: correct']}" + (f", because {NOT_RIGHT[(kind, n)]}" if (kind, n) in NOT_RIGHT else ""))
            print()

Name answers on which the judge and the hand check disagree = 5

name question 7: Which company, featured in multiple TechCrunch articles, is not only responsible for introducing GPT-4 Turbo and planning to make GPT-4 with vision available but is also predicted to dominate the AI tools platform market, overshadowing competitors like Hugging Face, and is known for creating the popular generative AI, ChatGPT?
  Correct answer: OpenAI
  Answer: OpenAI is the company responsible for introducing GPT-4 Turbo and planning to make GPT-4 with vision available, and is predicted to dominate the AI tools platform market, overshadowing competitors like Hugging Face. [2, 4, 7] [4]
  Judge: not right. Hand check: right

name question 14: Which social media platform, mentioned in articles from 'The Guardian' and 'TechCrunch', has been used for both the indirect delivery of pills and as a source of firsthand information regarding events in Gaza due to trust issues with other platforms?
  Correct answer

**Observation:** the judge ruled 6 answers *right* that are not:
- question 8 declines
- question 17 gives neither yes nor no
- questions 11, 13, 15 and 30 give the opposite of the correct answer

In each, the answer discusses what the question asks about without giving the correct answer. The judge also ruled 2 right answers *not right*, questions 10 and 19: both answer a question that asks whether nothing changed, and both say that it changed, which means no.

**Observation:** the judge ruled 5 name answers *not right* although each names the correct company, as the word match found. So the judge's *not right* does not mean the correct answer is missing from the answer.

**Conclusion:** on name questions, only the word match agrees with the hand check more often than ruling every answer *right*. On yes/no questions, only the judge does. Which checker to trust depends on how the answers are written, and only a hand check of a sample shows it.

## Step 3 of 4: faithfulness by judge

**Can the judge tell an answer taken from the chunks from one that is not?**

**[in real life]** The judge rules each name answer *faithful* or *not faithful*, given the same returned chunks the language model read, and its ruling is compared with the hand check's.

As in Step 2, the judge is compared with the blind checker: here it rules every answer *not faithful*, the hand check's more common ruling for faithfulness.

In [10]:
name_q["judge: faithful"] = ["faithful" if says_yes(common.generate(JUDGE_FAITHFUL.format(chunks=numbered(r), answer=a), max_tokens=10))
                             else "not faithful" for r, a in zip(name_q["returned"], name_q["answer_text"])]
always_no = (name_q["hand: faithful"] == "not faithful").sum()
print(f"name questions, blind checker, rules every answer not faithful: agreement with the hand check = {always_no} / {SAMPLE}")
print()
agree, share = agreement(name_q, "judge", "faithful")
print(f"name questions, judge: agreement with the hand check = {share}")
print()
show(against_hand(name_q, "judge", "faithful", ["faithful", "not faithful"]))

name questions, blind checker, rules every answer not faithful: agreement with the hand check = 18 / 30

name questions, judge: agreement with the hand check = 3 faithful + 15 not faithful = 18 / 30

                          judge: faithful  judge: not faithful  total
hand check: faithful                    3                    9     12
hand check: not faithful                3                   15     18
total                                   6                   24     30


**Observation:** the judge ruled 24 of the 30 answers *not faithful*, and the hand check ruled 9 of those 24 *faithful*. The blind checker, which rules every answer *not faithful*, agrees with the hand check exactly as often as the judge. So on these 30 answers, the judge's faithfulness rulings are no better than ruling every answer *not faithful* without reading.

<u>Note</u>: in Step 2, the judge replied *No* 18 times, a count unrelated to this step. In this step, three counts are 18:
- the answers the hand check ruled *not faithful*
- the blind checker's agreement
- the judge's agreement

The first two are equal by design: that checker agrees on exactly the answers the hand check ruled *not faithful*. The judge's 18 equals them by coincidence.

The output below lists the claim that made each answer *not faithful* in the hand check, with the question, the answer and the correct answer:

In [11]:
for n, claim in NOT_FAITHFUL.items():
    row = name_q.iloc[n - 1]
    print(f"name question {n}, the claim in no chunk: {claim}")
    print(f"  Question: {row['query']}")
    print(f"  Answer: {tidy(row['answer_text'])}")
    print(f"  Correct answer: {row['answer']}. Hand check: {row['hand: correct']}")
    print()
print("=" * 60)
print()
print("An example, the returned chunks of name question 4, none of which holds its claim:")
for i, c in enumerate(name_q.iloc[3]["returned"], start=1):
    print(f"[{i}] {c}")

name question 3, the claim in no chunk: that Google influences local search rankings with specific criteria
  Question: Which company, often covered by The Verge and TechCrunch, has been reported to make deals with major tech companies to maintain its default search engine status, influences local search rankings with specific criteria, and faces a class action antitrust suit for its impact on news publishers' content and revenue?
  Answer: Google is the company that has been reported to make deals with major tech companies to maintain its default search engine status, influences local search rankings with specific criteria, and faces a class action antitrust suit for its impact on news publishers' content and revenue. [1] [2] [3]
  Correct answer: Google. Hand check: right

name question 4, the claim in no chunk: that Apple enforces uniform terms through its store and payment system
  Question: Which company, covered by both The Verge and Engadget, is known for its privacy stance in a

**Observation:** all 18 answers are *not faithful* for the same reason: the language model copied a description from the question into its answer, and no returned chunk holds that description. For example, questions 4, 13 and 19 all describe a company that *"enforces uniform terms through its store and payment system"*, and all three answers repeat that description. So in these answers, the claims that are in no chunk come from the question, not from the language model's own invention.

**Conclusion:** checking faithfulness by hand takes reading every claim against every chunk, and this small judge cannot do it reliably. A faithfulness score from a judge has to be checked against a hand check, and against the blind checker, before it is trusted.

## Step 4 of 4: whose fault

**When an answer is not right, did the search return the material it needed?**

**[in real life]** Each question was written from two to four articles, and asks about a fact from each. For each answer, the check asks whether the returned chunks hold at least one listed sentence from every one of those articles. In the table, "every article found" means they do. The check looks for each listed sentence word for word.

When every article is found and the answer is not right, the language model failed. When an article is missing, its listed sentences were not in front of the language model, and rule 3 asks it to reply *"Not enough information"* when the chunks lack the answer. An answer that is not right is then likely a search failure. But another returned chunk may state the same fact in other words, so the check can blame the search wrongly. The answers that are right are counted too: a check that blames the search is only useful if right answers come with found articles more often than answers that are not right.

In [12]:
def every_article_found(row):
    sentences_of = defaultdict(list)
    for e in row["evidence_list"]:
        sentences_of[e["title"]].append(tidy(e["fact"]))
    return all(any(s in c for s in sentences for c in row["returned"]) for sentences in sentences_of.values())


rows = []
for found in [True, False]:
    for kind, table in [("name", name_q), ("yes/no", yes_no_q)]:
        part = table[table.apply(every_article_found, axis=1) == found]
        rows.append({"every article found": "yes" if found else "no", "kind of question": kind,
                     "questions": str(len(part)), "hand check: right": str((part["hand: correct"] == "right").sum()),
                     "hand check: not right": str((part["hand: correct"] == "not right").sum())})
both = pd.concat([name_q, yes_no_q])
rows.append({"every article found": "total", "kind of question": "",
             "questions": str(len(both)), "hand check: right": str((both["hand: correct"] == "right").sum()),
             "hand check: not right": str((both["hand: correct"] == "not right").sum())})
print("Answers by whether the returned chunks hold a listed sentence from every article the question needs:")
print()
show(pd.DataFrame(rows))

Answers by whether the returned chunks hold a listed sentence from every article the question needs:

every article found  kind of question  questions  hand check: right  hand check: not right
yes                  name                      2                  2                      0
yes                  yes/no                    3                  3                      0
no                   name                     28                 25                      3
no                   yes/no                   27                 13                     14
total                                         60                 43                     17


**Observation:** the search found every article for only 5 of the 60 questions, and all 5 answers are *right*. For the other 55, it missed at least one article, and those 55 hold all 17 answers that are not right, but also 25 + 13 = 38 answers that are right. 5 questions with every article found are too few to say how often the language model errs when it has every fact.

**Hypothesis:** an answer can be right with an article missing in three ways:
- Another returned chunk can state the missing fact in other words.
- For a name question, one article's facts can be enough to name the company.
- For a yes/no question, a reply of yes or no can be right by chance.

This module does not test which.

**Conclusion:** every answer that is not right came with an article missing from the returned chunks, but so did most right answers, so a missing article does not by itself explain an answer that is not right. On all the 60 questions, the check cannot separate a search failure from a language model failure: the search missed an article for almost every question.

## Key takeaways
- Correctness compares an answer with the correct answer, and faithfulness compares it with the returned chunks. The two differ: 27 of 30 name answers were right, but only 12 were faithful.
- A word match works when the answer writes the correct answer as a word, such as a name, and fails when the answer states it in other words, as most yes/no answers did.
- A language model as judge reads meaning, but errs in both directions, and gives no reason for a ruling.
- A checker's agreement with the hand check means something only next to the blind checker, which always gives the more common ruling. This judge beat the blind checker only on yes/no correctness: on name correctness and on faithfulness, it did no better.
- Every answer that was not right came with a search that missed an article, but so did most right answers. So a missed article alone does not show that the search caused an answer to be not right.
- These counts are rough, for two reasons:
  - one small language model both wrote the answers and judged them;
  - each kind of question has only 30 questions.

## Conclusion
An answer is graded on two measures, correctness and faithfulness, and every checker of either measure, such as the word match or the judge, has to be checked by hand on a sample before its rulings are trusted.